# 🪟 24-信号处理 · 05 窗函数与功率谱估计

> 关键词：窗函数、频谱泄漏、PSD、Welch
> §01 提过泄漏、§03 用了 Hamming——本章系统对比各窗函数，并把功率谱估计（PSD/Welch）讲透。

## 本节要解决的问题

| # | 问题 | 对应内容 |
| --- | --- | --- |
| 1 | 为什么 DFT 前必须加窗？ | §1 核心概念 |
| 2 | 各窗函数的主瓣/旁瓣怎么对比？ | §2 手算 ✍️ |
| 3 | 功率谱 PSD 和 |X[k]|² 有什么区别？ | §3 深入原理 |
| 4 | Welch 平均为什么能降方差？ | §4 代码实战 |

## 故事引入

一段 10 秒正弦，DFT 却出现了一堆"幽灵频谱"——不是信号的问题，是**截断**的问题：把无限长正弦硬切成有限段，相当于乘了矩形窗，频谱被窗函数卷积"抹糊"了。选对窗 = 选对"取景框"。

## 核心概念

**加窗的数学**：观测 $x[n]w[n]$ 的 DFT，频域是 $X(f)*W(f)$——频谱被窗的频谱卷积。矩形窗旁瓣高（−13dB），正弦会泄漏；余弦类窗旁瓣低。

**常用窗**（$N$ 点，$0\le n\le N-1$）：

- 矩形：$w[n]=1$，主瓣最窄，旁瓣 −13dB

| **Hann**：$w[n]=0.5[1-\cos(2\pi n/(N-1))]$，旁瓣 −31dB

- **Hamming**：$w[n]=0.54-0.46\cos(2\pi n/(N-1))$，旁瓣 −43dB（§03 用过）
- **Blackman**：$w[n]=0.42-0.5\cos(2\pi n/(N-1))+0.08\cos(4\pi n/(N-1))$，旁瓣 −58dB

**权衡**：旁瓣越低，主瓣越宽（频率分辨率越差）。

## ✍️ 手算窗函数数值与主瓣

**设定 A：Hann 窗（N=8）**

$$w[n]=0.5\left(1-\cos\frac{2\pi n}{7}\right)$$

$$w[0]=0.5(1-1)=\mathbf{0},\qquad w[3]=0.5\left(1-\cos\frac{6\pi}{7}\right)=0.5(1+0.9009)=\mathbf{0.9505}$$

窗两端为 0、中间最高——"把中间数据看得最重"。

**设定 B：矩形窗主瓣**

$N=8$ 矩形窗的 DFT：$W[k]=\sum_{n=0}^{7}e^{-j2\pi kn/8}$，其幅度 $|W[k]|=|\sin(\pi k)/\sin(\pi k/8)|$。

$$W[0]=8,\quad W[1]=0\ (\text{第一个零点})\Rightarrow\text{主瓣宽}=2\ \text{bin}$$

主瓣越窄 → 频率分辨率越高；但旁瓣 $\approx -13$dB → 泄漏严重。

**手算收获**：

- 主瓣宽度决定"两个相邻频率能否分开"（分辨率），旁瓣决定"强信号会否污染弱信号"（动态范围）
- Hann 主瓣 ≈ 矩形 2 倍宽（分辨率减半），换旁瓣 −31dB——**没有免费午餐**
- 窗函数在时域相乘 = 频域卷积：$X*W$ 的主瓣是"抹糊核"、旁瓣是"幽灵峰"
- 面试速答：测幅值精度用矩形（幅度矫正无损耗）、测弱信号旁瓣干扰小用 Blackman、通用语音用 Hamming/Hann

In [1]:
# ---- §2.1 验证：numpy 复算窗函数 ----
import numpy as np
N = 8
hann = 0.5 * (1 - np.cos(2*np.pi*np.arange(N)/(N-1)))
print('Hann(N=8): w[0] =', round(hann[0], 4), ', w[3] =', round(hann[3], 4))
assert abs(hann[0]) < 1e-9 and abs(hann[3] - 0.9505) < 1e-3

wrect = np.ones(8)
W = np.abs(np.fft.fft(wrect))
print('矩形窗频谱: W[0] =', W[0], ', W[1] =', round(W[1], 6), '（第一个零点）')
assert W[1] < 1e-9
print('\n✓ 与手算一致')


Hann(N=8): w[0] = 0.0 , w[3] = 0.9505
矩形窗频谱: W[0] = 8.0 , W[1] = 0.0 （第一个零点）

✓ 与手算一致


功率谱估计在真实信号上是什么样子？下图是对一段真实脑电（EEG）做频谱分析的结果：频率轴上能看到若干明显的峰——α 节律（约 8–13Hz）、β 节律（约 13–30Hz）等。这些峰的位置与宽度，正是我们在本节用周期图/Welch 方法估计 PSD 时要捕捉的目标。

![功率谱估计在真实信号上是什么样子？下图是对一段真实脑电（EE](../figures/eeg_powerspectrum.jpg)

> **怎么读**：图中把真实 EEG 的功率谱分解成若干钟形峰（每个峰用中心频率 f_i、幅度 H_i、宽度 B_i 描述）。峰越尖，说明该频段的节律越强。窗函数的选择会影响峰的展宽与旁瓣——这就是"窗函数=频域取景框"的实际效果。
>
> 📚 **图源**：Hu H, Pu Z & Wang P (2022) *A flexible and accurate method for EEG rhythms extraction based on circulant singular spectrum analysis*, PeerJ 10:e13096（开放获取，CC-BY 4.0）图 1。


## 深入原理：功率谱密度 PSD

**周期图（periodogram）**：

$$\hat P(f)=\frac{1}{N}|X(f)|^2$$

是 PSD 的估计，但方差大（≈ 与信号功率同量级）。

**Welch 平均**：把 N 点分成 K 段（每段 M 点、重叠 50%）、逐段加窗取周期图再平均

$$\hat P_{\text{Welch}}(f)=\frac{1}{K}\sum_{k=1}^{K}\hat P_k(f)$$

**方差降低**：K 段独立平均 → 方差降为 $1/K$（代价：频率分辨率降为 $f_s/M$）。

**归一化**：单边谱 ×2（除 DC 与 Nyquist），配合窗功率补偿 $\sum w^2$。

## 代码实战：Welch 平均降方差

白噪声的 PSD 理论上是常数 $\sigma^2$。单段周期图毛刺多、4 段平均更平滑——波动（相对标准误）接近理论 $1/\sqrt K$。

In [2]:
# ---- 周期图 vs Welch 平均 ----
import numpy as np
rng = np.random.default_rng(0)
x = rng.standard_normal(1000)
M = 200
psd_one = np.abs(np.fft.rfft(x[:M]))**2 / M
segs = x[:800].reshape(4, M)
psd_avg = (np.abs(np.fft.rfft(segs))**2 / M).mean(axis=0)
def rse(p): return p.std() / p.mean()
print('单段周期图 RSE = %.3f' % rse(psd_one))
print('4 段 Welch  RSE = %.3f（理论 1/√4 = 0.500）' % rse(psd_avg))
assert rse(psd_avg) < rse(psd_one)
print('✓ 分段平均显著降低 PSD 估计方差')


单段周期图 RSE = 0.999
4 段 Welch  RSE = 0.476（理论 1/√4 = 0.500）
✓ 分段平均显著降低 PSD 估计方差


## 面试考点

**Q1：矩形窗和 Hann 窗怎么选？**

需要最高频率分辨率且无大动态范围需求 → 矩形；需要抑制泄漏、检测弱信号 → Hann/Hamming；窗越长分辨率越高但平稳性假设越弱。

**Q2：加窗为什么改变幅值？**

余弦窗的 DC 增益 <1（Hann 增益 0.5）——做幅值测量要除以窗增益，做功率要除以 $\sum w^2$。

**Q3：Welch 为什么好？**

平均降方差 $1/K$；与整体 FFT 相比牺牲分辨率换稳定性——"谱估计"与"谱分析"（单帧看峰值）是两个场景。

**Q4：PSD 的量纲？**

$\text{V}^2/\text{Hz}$；宽带噪声压得越低，谱峰越突出——检测淹没在噪声里的正弦就靠它。

## 小结与下一章预告

- 窗函数 = 频域卷积的"取景框"：主瓣宽=分辨率、旁瓣高=泄漏
- Hann w[0]=0/w[3]=0.9505、矩形窗主瓣 2 bin：手算见真章
- Welch 分段平均把估计方差降到 1/K
- 下一章：从时域/频域上升到**复频域 z 变换**——系统分析的总框架

## ✅ 自测清单

- [ ] 能手算 Hann/Hamming 窗的几个数值并解释主瓣/旁瓣权衡
- [ ] 能说出矩形窗第一个零点与主瓣宽度
- [ ] 知道 PSD 与 |X|² 的区别、Welch 平均的方差收益
- [ ] 能回答"加窗改幅值为什么要补偿